# Question 2: Three Methods for Option Pricing

In practice, derivatives are priced using a variety of techniques. In this section, we price the
same European call option using three different approaches—Black-Scholes (closed-form), Monte
Carlo simulation, and a binomial lattice—and compare them.

The chosen stock for the European call option is Eli Lilly and Company (LLY).

## 2.0 Setup

In [ ]:
# Stock Ticker for Eli Lilly and Company (LLY)
ticker = "LLY"

# Stock price S0
# Most-current stock price (stock closed at $939.47 USD on April 10th, 2026)
stock_price = 939.47

# Risk-free rate r (annual % from FRED; decimal form for BS / MC formulas)
# Market Yield on U.S. Treasury Securities at 6-Month Constant Maturity (as of April 9th, 2026)
# https://fred.stlouisfed.org/series/DGS6MO
risk_free_rate_pct = 3.71
r = risk_free_rate_pct / 100.0

# At-the-money implied volatility σ from the Q1 volatility surface for LLY
implied_volatility = 0.2690

# At-the-money European call, 6 months to expiry (used in Problems 2.1–2.3)
time_to_expiry = 0.5  # years
strike_price = stock_price  # K = S0 (ATM)

# Monte Carlo (Problem 2.2): Euler–Maruyama time steps (∆t = T / n_steps)
mc_time_steps = 126
mc_seed = 42  # reproducible terminal-stock simulations

## 2.1 Black-Scholes
(a) Compute the Black-Scholes price using the formula from Problem 1. This is your benchmark.

In [ ]:
import numpy as np
from scipy.stats import norm


def bs_call(S, K, T, r, sigma):
    """
    Price the call option for stock S using the Black-Scholes European call
    formula from Question 1 (a)
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


# Define variables
S0 = stock_price  # Most-recent stock price
K = strike_price  # Strike price (K = S0 for an at-the-money call)
T = time_to_expiry  # Time to expiry (6 months as specified)
sigma = implied_volatility  # At-the-money implied volatility (from Question 1)

bs_benchmark_price = bs_call(S0, K, T, r, sigma)

print("Question 2.1: Black-Scholes benchmark (European ATM call for LLY)")
print(f"  LLY spot S0     : ${S0:,.2f}")
print(f"  Strike K        : ${K:,.2f}  (ATM: K = S0)")
print(f"  Time to expiry T: {T} years (6 months)")
print(f"  Risk-free r     : {r:.4f} ({risk_free_rate_pct}% p.a.)")
print(f"  Implied vol σ   : {sigma:.4f}")
print()
print(f"  Black-Scholes call price: ${bs_benchmark_price:,.4f}")

Question 2.1: Black-Scholes benchmark (European ATM call for LLY)
  LLY spot S0     : $939.47
  Strike K        : $939.47  (ATM: K = S0)
  Time to expiry T: 0.5 years (6 months)
  Risk-free r     : 0.0371 (3.71% p.a.)
  Implied vol σ   : 0.2690

  Black-Scholes call price: $79.4962


## 2.2 Monte Carlo

Under the risk-neutral measure, simulate geometric Brownian motion with **Euler–Maruyama** using
**n = 126** steps ($\Delta t = T/n$). The Monte Carlo estimate is
$\hat{C} = e^{-rT}\frac{1}{N}\sum_i \max(S_T^{(i)}-K,0)$ with standard error
$\mathrm{SE} = \hat{\sigma}_{\text{payoff}}/\sqrt{N}$ and **95%** interval $\hat{C} \pm 1.96\,\mathrm{SE}$.

**(a)** Convergence plot for $N$ from 100 to 500,000; MC price $\pm 2$ standard errors vs. Black–Scholes; smallest $N$ (on the grid) within **1%** of BS; table of $\hat{C}$, SE, and 95% CI at each $N$; discussion of convergence to BS.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

# --- Risk-neutral Euler–Maruyama to terminal S_T (assignment: n = 126, Δt = T/n) ---


def simulate_terminal_prices_gbm(S0, r, sigma, T, n_steps, n_paths, rng):
    """Simulate GBM under Q: dS = r S dt + σ S dW. Returns S_T for each path."""
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    S = np.full(n_paths, S0, dtype=np.float64)
    for _ in range(n_steps):
        Z = rng.standard_normal(n_paths)
        S = S + r * S * dt + sigma * S * sqrt_dt * Z
    return S


rng = np.random.default_rng(mc_seed)

N_max = 500_000
n_steps = mc_time_steps

t0 = time.perf_counter()
S_T = simulate_terminal_prices_gbm(S0, r, sigma, T, n_steps, N_max, rng)
t_sim = time.perf_counter() - t0

# Discounted payoffs (same discounting as in the assignment)
payoff = np.exp(-r * T) * np.maximum(S_T - K, 0.0)

# Path counts N ∈ [100, 500_000] (dense grid for convergence + table)
N_list = np.unique(
    np.clip(
        np.round(np.logspace(2, np.log10(N_max), 20)).astype(int),
        100,
        N_max,
    )
)
if N_list[0] < 100:
    N_list = np.unique(np.concatenate([[100], N_list]))
if N_list[-1] < N_max:
    N_list = np.unique(np.concatenate([N_list, [N_max]]))

rows = []
mc_prices = []
se_list = []

for N in N_list:
    x = payoff[:N]
    c_hat = float(np.mean(x))
    se = float(np.std(x, ddof=1) / np.sqrt(N)) if N > 1 else 0.0
    ci_lo = c_hat - 1.96 * se
    ci_hi = c_hat + 1.96 * se
    rows.append(
        {
            "N": N,
            "MC_price": c_hat,
            "SE": se,
            "CI_95_low": ci_lo,
            "CI_95_high": ci_hi,
            "rel_err_vs_BS": abs(c_hat - bs_benchmark_price) / bs_benchmark_price,
        }
    )
    mc_prices.append(c_hat)
    se_list.append(se)

mc_table = pd.DataFrame(rows)

# Smallest N on this grid within 1% of Black–Scholes (relative error on point estimate)
within_1pct = mc_table[mc_table["rel_err_vs_BS"] <= 0.01]
N_within_1pct = int(within_1pct["N"].iloc[0]) if len(within_1pct) else None

print("Question 2.2: Monte Carlo (Euler–Maruyama, n = 126 steps, Δt = T/n)")
print(
    f"  Simulated {N_max:,} terminal prices in {t_sim:.3f} s (one stream, nested N for convergence)."
)
print()
if N_within_1pct is not None:
    print(f"  Smallest N on this grid with |MC − BS| / BS ≤ 1%: N = {N_within_1pct:,}")
else:
    print("  No N on this grid reaches |MC − BS| / BS ≤ 1% (try larger N_max).")
print()
print("  MC price, SE, and 95% CI at each N (grid):")
print(mc_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("  (rel_err_vs_BS = |MC_price − BS| / BS_benchmark_price)")
print()

# ---- Convergence plot: MC ± 2 SE vs BS ----
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axhline(
    bs_benchmark_price,
    color="black",
    linestyle="-",
    linewidth=1.5,
    label="Black–Scholes (benchmark)",
)
mc_arr = np.array(mc_prices)
se_arr = np.array(se_list)
ax.semilogx(N_list, mc_arr, color="C0", label=r"$\hat{C}$ (Monte Carlo)")
ax.fill_between(
    N_list,
    mc_arr - 2 * se_arr,
    mc_arr + 2 * se_arr,
    color="C0",
    alpha=0.25,
    label=r"$\hat{C} \pm 2\,\mathrm{SE}$",
)
ax.set_xlabel("Number of paths $N$")
ax.set_ylabel("Option price")
ax.set_title("Question 2.2: MC convergence vs. Black–Scholes (LLY ATM call, 6M)")
ax.legend(loc="upper right")
ax.grid(True, which="both", alpha=0.35)
plt.tight_layout()
plt.show()

print("Interpretation:")
print(
    "  • The MC estimate approaches the BS price as N grows (law of large numbers); it does not"
)
print(
    "    match BS exactly for any finite N because of sampling noise—only in the limit N → ∞."
)
print(
    "  • With nested prefixes of the same random stream, the MC estimate vs. N is typical"
)
print("    of a Monte Carlo standard error shrinking as 1/√N.")